Part A: Intersections & Vertex Enumeration

In [ ]:
def solve_linear_system(A_sub, b_sub):
    """Solves a small 4x4 linear system using basic Gaussian Elimination."""
    n = len(b_sub)
    # Create augmented matrix
    M = [A_sub[i] + [b_sub[i]] for i in range(n)]
    
    for i in range(n):
        # Find pivot
        if abs(M[i][i]) < 1e-9:
            for k in range(i + 1, n):
                if abs(M[k][i]) > 1e-9:
                    M[i], M[k] = M[k], M[i]
                    break
            else:
                return None # Singular matrix / No unique intersection
        
        # Normalize pivot row
        pivot = M[i][i]
        M[i] = [val / pivot for val in M[i]]
        
        # Eliminate column entries
        for j in range(n):
            if j != i:
                factor = M[j][i]
                M[j] = [M[j][k] - factor * M[i][k] for k in range(n + 1)]
                
    return [row[-1] for row in M]

def intersections(A, b):
    """Finds all feasible vertices by intersecting combinations of constraints."""
    import itertools
    num_constraints = len(A)
    num_vars = 4
    feasible_vertices = []
    
    # Try every unique combination of 4 constraint hyperplanes
    for indices in itertools.combinations(range(num_constraints), num_vars):
        A_sub = [A[i] for i in indices]
        b_sub = [b[i] for i in indices]
        
        point = solve_linear_system(A_sub, b_sub)
        if point is not None:
            # Verify feasibility against ALL constraints
            is_feasible = True
            for i in range(num_constraints):
                lhs = sum(A[i][j] * point[j] for j in range(num_vars))
                if lhs > b[i] + 1e-5: # Tolerance for floating-point noise
                    is_feasible = False
                    break
            
            if is_feasible:
                # Deduplicate points
                if not any(sum((point[k] - v[k])**2 for k in range(num_vars)) < 1e-5 for v in feasible_vertices):
                    feasible_vertices.append(point)
                    
    return feasible_vertices

def optimizerLP_vertex(A, b, c):
    """Finds the optimal point by evaluating the cost at all feasible vertices."""
    vertices = intersections(A, b)
    if not vertices:
        return None, float('inf')
        
    best_point = None
    min_cost = float('inf')
    
    for v in vertices:
        cost = sum(c[i] * v[i] for i in range(len(c)))
        if cost < min_cost:
            min_cost = cost
            best_point = v
            
    return best_point, min_cost

Part B: The Branch and Bound Global Wrapper

In [ ]:
def is_integer(val, tol=1e-5):
    return abs(val - round(val)) < tol

def optimizerIPBB(A, b, c):
    """Global Integer Programming Routine using Branch and Bound."""
    
    # Track the globally verified best integer allocation
    best_allocation = {"point": None, "cost": float('inf')}
    
    def branch_and_bound_node(current_A, current_b):
        # 1. Solve relaxed continuous LP at current node
        point, cost = optimizerLP_vertex(current_A, current_b, c)
        
        # Prune if the node is mathematically infeasible or worse than our current best floor
        if point is None or cost >= best_allocation["cost"]:
            return
            
        # 2. Inspect results for non-integer fractional variables
        fractional_var_idx = -1
        for i, val in enumerate(point):
            if not is_integer(val):
                fractional_var_idx = i
                break
                
        # 3. If all variables are perfectly whole integers, update upper bound bound
        if fractional_var_idx == -1:
            if cost < best_allocation["cost"]:
                best_allocation["cost"] = cost
                best_allocation["point"] = [round(v) for v in point]
            return
            
        # 4. Branching phase: Create floor and ceiling constraint partitions
        val_to_branch = point[fractional_var_idx]
        floor_val = int(val_to_branch)
        ceil_val = floor_val + 1
        
        # Construct Left Branch (x_i <= floor_val)
        new_row_left = [0] * 4
        new_row_left[fractional_var_idx] = 1
        branch_and_bound_node(current_A + [new_row_left], current_b + [floor_val])
        
        # Construct Right Branch (x_i >= ceil_val -> -x_i <= -ceil_val)
        new_row_right = [0] * 4
        new_row_right[fractional_var_idx] = -1
        branch_and_bound_node(current_A + [new_row_right], current_b + [-ceil_val])

    # Run from baseline system state
    branch_and_bound_node(A, b)
    return best_allocation["point"], best_allocation["cost"]

Task 5: Running & Plotting the Output

In [ ]:
# System Parameters setup matching standard matrices
A_base = [
    [1, 1, 0, 0], [-1, -1, 0, 0], [0, 0, 1, 1], [0, 0, -1, -1],
    [1, 0, 1, 0], [0, 1, 0, 1],
    [1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 1, 0], [0, 0, 0, 1],
    [-1, 0, 0, 0], [0, -1, 0, 0], [0, 0, -1, 0], [0, 0, 0, -1]
]
b_base = [60, -60, 40, -40, 80, 40, 50, 50, 50, 50, 0, 0, 0, 0]
c_base = [100, 80, 90, 120]

optimal_pt, optimal_val = optimizerIPBB(A_base, b_base, c_base)

print(f"Optimal Integer Allocation Vector: {optimal_pt}")
print(f"Minimum Total Transportation Cost: {optimal_val} SAR")

In [ ]:
import matplotlib.pyplot as plt

# Generate evaluation coordinate span
x_vals = range(0, 65)
y_line_eq = [60 - x for x in x_vals]  # From exact constraint x11 + x12 = 60

plt.figure(figsize=(8, 5))
plt.plot(x_vals, y_line_eq, label='Port 1 Allocation Line ($x_{11} + x_{12} = 60$)', color='blue', linewidth=2)

# Shade the physical route bounds
plt.axvline(50, color='red', linestyle='--', label='Max Bus Route Cap ($x_{ij} \leq 50$)')
plt.axhline(50, color='red', linestyle='--')

# Highlight the optimal integer coordinate pinpoint
plt.scatter(optimal_pt[0], optimal_pt[1], color='gold', edgecolor='black', s=150, zorder=5, label=f'Optimal Point {optimal_pt[:2]}')

plt.title('Hajj Logistics Feasible Target Sub-Region (Port 1 Variables)')
plt.xlabel('Pilgrims assigned from Port 1 to Hotel 1 ($x_{11}$)')
plt.ylabel('Pilgrims assigned from Port 1 to Hotel 2 ($x_{12}$)')
plt.xlim(0, 70)
plt.ylim(0, 70)
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()